# Gages Activity Geodatabase creation

In [6]:
import pandas as pd
import geopandas as gpd
import shapely
import pyogrio

print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)
print("shapely:", shapely.__version__)
print("pyogrio:", pyogrio.__version__)

print("\nGDAL drivers:")
print(pyogrio.list_drivers())

pandas: 3.0.5
geopandas: 1.1.3
shapely: 2.1.2
pyogrio: 0.10.0

GDAL drivers:
{'MEM': 'rw', 'PCIDSK': 'rw', 'PDS4': 'rw', 'VICAR': 'rw', 'MBTiles': 'rw', 'EEDA': 'r', 'OGCAPI': 'r', 'GDALG': 'r', 'ESRI Shapefile': 'rw', 'MapInfo File': 'rw', 'LVBAG': 'r', 'S57': 'rw', 'DGN': 'rw', 'OGR_VRT': 'r', 'CSV': 'rw', 'NAS': 'r', 'GML': 'rw', 'GPX': 'rw', 'LIBKML': 'rw', 'KML': 'rw', 'GeoJSON': 'rw', 'GeoJSONSeq': 'rw', 'ESRIJSON': 'r', 'TopoJSON': 'r', 'Interlis 1': 'r', 'Interlis 2': 'r', 'OGR_GMT': 'rw', 'GPKG': 'rw', 'SQLite': 'rw', 'WAsP': 'rw', 'OpenFileGDB': 'rw', 'DXF': 'rw', 'CAD': 'r', 'FlatGeobuf': 'rw', 'GeoRSS': 'rw', 'VFK': 'r', 'PGDUMP': 'rw', 'OSM': 'r', 'GPSBabel': 'rw', 'OGR_PDS': 'r', 'WFS': 'r', 'OAPIF': 'r', 'EDIGEO': 'r', 'Idrisi': 'r', 'ODS': 'rw', 'XLSX': 'rw', 'Elasticsearch': 'rw', 'Carto': 'rw', 'AmigoCloud': 'rw', 'SXF': 'r', 'Selafin': 'rw', 'JML': 'rw', 'PLSCENES': 'r', 'CSW': 'r', 'VDV': 'rw', 'GMLAS': 'r', 'MVT': 'rw', 'NGW': 'rw', 'MapML': 'rw', 'GTFS': 'r', 'PMT

In [8]:
import pandas as pd
from pathlib import Path

# Update this if your CSV is stored somewhere else
csv_path = Path(
    "/Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/intermediate/gages_activity_with_api_locations_recovered.csv"
)

df = pd.read_csv(csv_path, low_memory=False)

print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nState names:")
display(df["state_name"].value_counts().head(20))

Rows and columns: (23242, 13)

Column names:
['site', 'n_years_active', 'earliest_active_year', 'which_years_active', 'any_gaps', 'gap_years', 'api_id', 'station_name', 'state_name', 'latitude', 'longitude', 'location_found', 'coordinates_available']

First 5 rows:


,site,n_years_active,earliest_active_year,which_years_active,any_gaps,gap_years,api_id,station_name,state_name,latitude,longitude,location_found,coordinates_available
0,7010000,159,1863,1863;1864;1865;1866;1867;1868;1869;1870;1871;1...,False,NaN,USGS-07010000,"Mississippi River at St. Louis, MO",Missouri,38.629000,-90.179778,True,True
1,5420500,148,1874,1874;1875;1876;1877;1878;1879;1880;1881;1882;1...,False,NaN,USGS-05420500,"Mississippi River at Clinton, IA",Iowa,41.780586,-90.252073,True,True
2,3568000,134,1875,1875;1876;1877;1878;1879;1880;1881;1882;1883;1...,False,NaN,USGS-03568000,"TENNESSEE RIVER AT CHATTANOOGA, TN",Tennessee,35.086776,-85.278430,True,True
3,3193000,144,1878,1878;1879;1880;1881;1882;1883;1884;1885;1886;1...,False,NaN,USGS-03193000,"KANAWHA RIVER AT KANAWHA FALLS, WV",West Virginia,38.138163,-81.214274,True,True
4,5474500,143,1879,1879;1880;1881;1882;1883;1884;1885;1886;1887;1...,False,NaN,USGS-05474500,"Mississippi River at Keokuk, IA",Iowa,40.393655,-91.374318,True,True



State names:


state_name
California        2327
Colorado          1160
Florida            961
Washington         941
Texas              922
Montana            710
Utah               698
Idaho              693
Oregon             669
New York           652
Wisconsin          645
Pennsylvania       624
Wyoming            601
North Carolina     540
Ohio               499
Arizona            488
Georgia            481
Alaska             474
Tennessee          431
Michigan           421
Name: count, dtype: int64

In [9]:
required_columns = [
    "site",
    "state_name",
    "latitude",
    "longitude",
    "earliest_active_year",
    "which_years_active"
]

missing = [col for col in required_columns if col not in df.columns]

if missing:
    print("Missing required columns:", missing)
else:
    print("All required columns are present.")

    print("\nMissing coordinates:")
    print(df[["latitude", "longitude"]].isna().sum())

    print("\nEarliest active years:")
    print(df["earliest_active_year"].describe())

All required columns are present.

Missing coordinates:
latitude     3
longitude    3
dtype: int64

Earliest active years:
count    23242.000000
mean      1964.992987
std         30.864613
min       1863.000000
25%       1941.000000
50%       1965.000000
75%       1990.000000
max       2021.000000
Name: earliest_active_year, dtype: float64


In [10]:

# States to export
target_states = [
    "Nebraska",
    "Iowa",
    "Missouri",
    "Arkansas",
    "Oklahoma",
    "Indiana",
    "Kentucky",
    "West Virginia",
    "Louisiana",
    "Alabama",
    "Tennessee",
]

# Check which requested states are present
print("Requested states found:")
print(
    df.loc[df["state_name"].isin(target_states), "state_name"]
      .value_counts()
      .reindex(target_states, fill_value=0)
)

# Inspect activity-year strings
for i, value in df["which_years_active"].dropna().head(5).items():
    years = [int(year) for year in str(value).split(";") if year.strip()]
    print(f"Row {i}: first years={years[:5]}, last years={years[-3:]}")

# Check missing coordinates by record
print("\nRows missing coordinates:")
print(
    df.loc[
        df["latitude"].isna() | df["longitude"].isna(),
        ["site", "station_name", "state_name", "latitude", "longitude"]
    ]
)


Requested states found:
state_name
Nebraska         319
Iowa             252
Missouri         397
Arkansas         260
Oklahoma         339
Indiana          368
Kentucky         321
West Virginia    288
Louisiana        195
Alabama          361
Tennessee        431
Name: count, dtype: int64
Row 0: first years=[1863, 1864, 1865, 1866, 1867], last years=[2019, 2020, 2021]
Row 1: first years=[1874, 1875, 1876, 1877, 1878], last years=[2019, 2020, 2021]
Row 2: first years=[1875, 1876, 1877, 1878, 1879], last years=[2006, 2007, 2008]
Row 3: first years=[1878, 1879, 1880, 1881, 1882], last years=[2019, 2020, 2021]
Row 4: first years=[1879, 1880, 1881, 1882, 1883], last years=[2019, 2020, 2021]

Rows missing coordinates:
                  site                                     station_name  \
10535         11516100  GREEN SPRINGS POWERPLANT DIVER NR ASHLAND,OREG.   
22351  421247071470201                    BRIDGE-RT20W-QUINSIGAMOND, MA   
22353  422108071052501                          BRI

In [15]:

from pathlib import Path
import pandas as pd
import geopandas as gpd
import pyogrio

# Output folder: Documents/Gages_State_GDBs
output_folder = Path.home() / "/Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/processed" / "Gages_State_GDBs"
output_folder.mkdir(parents=True, exist_ok=True)

target_states = [
    "Nebraska", "Iowa", "Missouri", "Arkansas", "Oklahoma",
    "Indiana", "Kentucky", "West Virginia", "Louisiana",
    "Alabama", "Tennessee"
]

snapshot_years = list(range(1860, 2030, 10))  # 1860 through 2020

# Work on a copy; do not change the original dataframe
work = df.copy()

# Convert coordinates and earliest year to numeric
work["latitude"] = pd.to_numeric(work["latitude"], errors="coerce")
work["longitude"] = pd.to_numeric(work["longitude"], errors="coerce")
work["earliest_active_year"] = pd.to_numeric(
    work["earliest_active_year"], errors="coerce"
)

# Parse semicolon-separated activity years into sets of integers
def parse_active_years(value):
    if pd.isna(value):
        return set()
    return {
        int(year.strip())
        for year in str(value).split(";")
        if year.strip().isdigit()
    }

work["_active_years"] = work["which_years_active"].apply(parse_active_years)

# Keep target states with valid point coordinates
valid_coordinates = (
    work["latitude"].between(-90, 90)
    & work["longitude"].between(-180, 180)
)

work = work[
    work["state_name"].isin(target_states)
    & valid_coordinates
].copy()

print("Output folder:", output_folder)
print("Records with valid coordinates in target states:", len(work))
print("States represented:", work["state_name"].nunique())
print("Snapshot years:", snapshot_years)
print("pyogrio version:", pyogrio.__version__)
print("OpenFileGDB driver available:",
      "OpenFileGDB" in pyogrio.list_drivers().index)


Output folder: /Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/processed/Gages_State_GDBs
Records with valid coordinates in target states: 3531
States represented: 11
Snapshot years: [1860, 1870, 1880, 1890, 1900, 1910, 1920, 1930, 1940, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020]
pyogrio version: 0.10.0


AttributeError: 'dict' object has no attribute 'index'

In [16]:

import shutil
import pandas as pd
import geopandas as gpd
import pyogrio

# These paths are safe to clean up because the previous export
# stopped during creation of the first output geodatabase.
gdb_paths = {
    state: output_folder / f"{state.replace(' ', '_')}.gdb"
    for state in target_states
}

for path in gdb_paths.values():
    if path.exists():
        print("Removing partial output:", path.name)
        shutil.rmtree(path)

layer_counts = []

for state in target_states:
    state_df = work[work["state_name"] == state].copy()
    gdb_path = gdb_paths[state]

    print(f"\nCreating {gdb_path.name} ({len(state_df)} source gages)")

    for i, year in enumerate(snapshot_years):
        eligible = state_df[
            state_df["earliest_active_year"].notna()
            & (state_df["earliest_active_year"] <= year)
        ].copy()

        eligible["activity_status"] = eligible["_active_years"].apply(
            lambda years: "Active" if year in years else "Inactive"
        )
        eligible["snapshot_year"] = year
        eligible = eligible.drop(columns=["_active_years"])

        gdf = gpd.GeoDataFrame(
            eligible,
            geometry=gpd.points_from_xy(
                eligible["longitude"],
                eligible["latitude"]
            ),
            crs="EPSG:4326"
        )

        layer_name = f"Gages_{year}"

        # Explicit Point geometry allows even an empty snapshot layer.
        gdf.to_file(
            gdb_path,
            layer=layer_name,
            driver="OpenFileGDB",
            engine="pyogrio",
            geometry_type="Point",
            mode="w" if i == 0 else "a"
        )

        layer_counts.append({
            "state": state,
            "year": year,
            "layer": layer_name,
            "feature_count": len(gdf),
            "active_count": int(
                (gdf["activity_status"] == "Active").sum()
            ),
            "inactive_count": int(
                (gdf["activity_status"] == "Inactive").sum()
            )
        })

    print(f"Finished {state}")

counts_df = pd.DataFrame(layer_counts)

print("\nExport loop finished.")
print("Geodatabases:", len(target_states))
print("Layers attempted:", len(counts_df))
display(counts_df.head(20))



Creating Nebraska.gdb (319 source gages)
Finished Nebraska

Creating Iowa.gdb (252 source gages)


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field site of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field earliest_active_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field snapshot_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation

Finished Iowa

Creating Missouri.gdb (397 source gages)
Finished Missouri

Creating Arkansas.gdb (260 source gages)
Finished Arkansas

Creating Oklahoma.gdb (339 source gages)


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field site of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field earliest_active_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field snapshot_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation

Finished Oklahoma

Creating Indiana.gdb (368 source gages)
Finished Indiana

Creating Kentucky.gdb (321 source gages)
Finished Kentucky


...


Creating West_Virginia.gdb (288 source gages)
Finished West Virginia

Creating Louisiana.gdb (195 source gages)


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field site of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field earliest_active_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field snapshot_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation

Finished Louisiana

Creating Alabama.gdb (361 source gages)
Finished Alabama

Creating Tennessee.gdb (431 source gages)
Finished Tennessee

Export loop finished.
Geodatabases: 11
Layers attempted: 187


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field site of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field earliest_active_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field snapshot_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation

,state,year,layer,feature_count,active_count,inactive_count
0,Nebraska,1860,Gages_1860,0,0,0
1,Nebraska,1870,Gages_1870,0,0,0
2,Nebraska,1880,Gages_1880,0,0,0
3,Nebraska,1890,Gages_1890,0,0,0
4,Nebraska,1900,Gages_1900,1,0,1
5,Nebraska,1910,Gages_1910,1,0,1
6,Nebraska,1920,Gages_1920,3,0,3
7,Nebraska,1930,Gages_1930,17,15,2
8,Nebraska,1940,Gages_1940,63,56,7
9,Nebraska,1950,Gages_1950,150,138,12


In [18]:

for state in target_states:
    gdb_path = output_folder / f"{state.replace(' ', '_')}.gdb"

    if not gdb_path.exists():
        print(f"MISSING: {gdb_path.name}")
        continue

    layers = pyogrio.list_layers(gdb_path)
    print(f"{state}: {len(layers)} layers")


Nebraska: 17 layers
Iowa: 17 layers
Missouri: 17 layers
Arkansas: 17 layers
Oklahoma: 17 layers
Indiana: 17 layers
Kentucky: 17 layers
West Virginia: 17 layers
Louisiana: 17 layers
Alabama: 17 layers
Tennessee: 17 layers


In [19]:
from pathlib import Path
import shutil
import pandas as pd
import geopandas as gpd
import pyogrio

# Output folder
output_folder = (
    Path.home()
    / "Documents" / "GitHub" / "USGS-Gages-Research"
    / "data" / "processed" / "Gages_State_GDBs"
)
output_folder.mkdir(parents=True, exist_ok=True)

# Source coordinate CRS: longitude/latitude, as assumed in the existing workflow
SOURCE_CRS = "EPSG:4326"

# One NAD 83 / UTM projected CRS per state; units are meters
state_crs = {
    "Nebraska": "EPSG:26914",
    "Iowa": "EPSG:26915",
    "Missouri": "EPSG:26915",
    "Arkansas": "EPSG:26915",
    "Oklahoma": "EPSG:26914",
    "Indiana": "EPSG:26916",
    "Kentucky": "EPSG:26916",
    "West Virginia": "EPSG:26917",
    "Louisiana": "EPSG:26915",
    "Alabama": "EPSG:26916",
    "Tennessee": "EPSG:26916",
}

# Reuse your existing state and year lists
snapshot_years = list(range(1860, 2030, 10))

# Prepare data
work = df.copy()
work["latitude"] = pd.to_numeric(work["latitude"], errors="coerce")
work["longitude"] = pd.to_numeric(work["longitude"], errors="coerce")
work["earliest_active_year"] = pd.to_numeric(
    work["earliest_active_year"], errors="coerce"
)

def parse_active_years(value):
    if pd.isna(value):
        return set()
    return {
        int(year.strip())
        for year in str(value).split(";")
        if year.strip().isdigit()
    }

work["_active_years"] = work["which_years_active"].apply(
    parse_active_years
)

valid_coordinates = (
    work["latitude"].between(-90, 90)
    & work["longitude"].between(-180, 180)
)

work = work[
    work["state_name"].isin(state_crs)
    & valid_coordinates
].copy()

# Remove existing outputs only after confirming the folder is correct
gdb_paths = {
    state: output_folder / f"{state.replace(' ', '_')}.gdb"
    for state in state_crs
}

for path in gdb_paths.values():
    if path.exists():
        shutil.rmtree(path)

layer_counts = []

for state, target_crs in state_crs.items():
    state_df = work[work["state_name"] == state].copy()
    gdb_path = gdb_paths[state]

    print(f"\nCreating {state}.gdb in {target_crs}")

    for i, year in enumerate(snapshot_years):
        # A gage first becomes eligible in its earliest recorded active year
        eligible = state_df[
            state_df["earliest_active_year"].notna()
            & (state_df["earliest_active_year"] <= year)
        ].copy()

        eligible["activity_status"] = eligible["_active_years"].apply(
            lambda years: "Active" if year in years else "Inactive"
        )
        eligible["snapshot_year"] = year

        eligible = eligible.drop(columns=["_active_years"])

        # Build points in the source CRS, then project to NAD 83 UTM
        gdf = gpd.GeoDataFrame(
            eligible,
            geometry=gpd.points_from_xy(
                eligible["longitude"], eligible["latitude"]
            ),
            crs=SOURCE_CRS,
        )

        gdf = gdf.to_crs(target_crs)

        # Keep station identifiers as text
        if "site" in gdf.columns:
            gdf["site"] = gdf["site"].astype("string")
        if "api_id" in gdf.columns:
            gdf["api_id"] = gdf["api_id"].astype("string")

        layer_name = f"Gages_{year}"

        gdf.to_file(
            gdb_path,
            layer=layer_name,
            driver="OpenFileGDB",
            engine="pyogrio",
            geometry_type="Point",
            mode="w" if i == 0 else "a",
        )

        layer_counts.append({
            "state": state,
            "crs": target_crs,
            "year": year,
            "layer": layer_name,
            "feature_count": len(gdf),
            "active_count": int(
                (gdf["activity_status"] == "Active").sum()
            ),
            "inactive_count": int(
                (gdf["activity_status"] == "Inactive").sum()
            ),
        })

    print(f"Finished {state}")

counts_df = pd.DataFrame(layer_counts)

print("\nExport finished.")
print("Geodatabases:", len(gdb_paths))
print("Layers written:", len(counts_df))
display(counts_df.head(20))


Creating Nebraska.gdb in EPSG:26914


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field earliest_active_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field snapshot_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use laye

Finished Nebraska

Creating Iowa.gdb in EPSG:26915
Finished Iowa

Creating Missouri.gdb in EPSG:26915


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field earliest_active_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field snapshot_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use laye

Finished Missouri

Creating Arkansas.gdb in EPSG:26915
Finished Arkansas


...

Finished Louisiana

Creating Alabama.gdb in EPSG:26916


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field earliest_active_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field snapshot_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use laye

Finished Alabama

Creating Tennessee.gdb in EPSG:26916
Finished Tennessee

Export finished.
Geodatabases: 11
Layers written: 187


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field earliest_active_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field snapshot_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use laye

,state,crs,year,layer,feature_count,active_count,inactive_count
0,Nebraska,EPSG:26914,1860,Gages_1860,0,0,0
1,Nebraska,EPSG:26914,1870,Gages_1870,0,0,0
2,Nebraska,EPSG:26914,1880,Gages_1880,0,0,0
3,Nebraska,EPSG:26914,1890,Gages_1890,0,0,0
4,Nebraska,EPSG:26914,1900,Gages_1900,1,0,1
5,Nebraska,EPSG:26914,1910,Gages_1910,1,0,1
6,Nebraska,EPSG:26914,1920,Gages_1920,3,0,3
7,Nebraska,EPSG:26914,1930,Gages_1930,17,15,2
8,Nebraska,EPSG:26914,1940,Gages_1940,63,56,7
9,Nebraska,EPSG:26914,1950,Gages_1950,150,138,12


In [20]:
for state, gdb_path in gdb_paths.items():
    layers = pyogrio.list_layers(gdb_path)
    test_layer = "Gages_2020"

    check = gpd.read_file(
        gdb_path,
        layer=test_layer,
        engine="pyogrio",
    )

    print(
        f"{state}: {len(layers)} layers | "
        f"CRS: {check.crs} | "
        f"units: {check.crs.axis_info[0].unit_name if check.crs else 'unknown'}"
    )

Nebraska: 17 layers | CRS: EPSG:26914 | units: metre
Iowa: 17 layers | CRS: EPSG:26915 | units: metre
Missouri: 17 layers | CRS: EPSG:26915 | units: metre
Arkansas: 17 layers | CRS: EPSG:26915 | units: metre
Oklahoma: 17 layers | CRS: EPSG:26914 | units: metre
Indiana: 17 layers | CRS: EPSG:26916 | units: metre
Kentucky: 17 layers | CRS: EPSG:26916 | units: metre
West Virginia: 17 layers | CRS: EPSG:26917 | units: metre
Louisiana: 17 layers | CRS: EPSG:26915 | units: metre
Alabama: 17 layers | CRS: EPSG:26916 | units: metre
Tennessee: 17 layers | CRS: EPSG:26916 | units: metre
